In [ ]:
import numpy as np
import random

# 第16章 DDPG算法

## 目录
1. DDPG原理
2. 目标网络
3. 编程题

---
## 1. DDPG原理

In [ ]:
"""DDPG (Deep Deterministic Policy Gradient)"""
class DDPG:
    """DDPG算法"""
    def __init__(self, n_states, n_actions):
        self.actor = np.zeros(n_states)
        self.critic = np.zeros((n_states, n_actions))
        self.target_actor = self.actor.copy()
        self.target_critic = self.critic.copy()
        self.tau = 0.005
    
    def get_action(self, state):
        return np.tanh(self.actor[0] * state[0])
    
    def soft_update(self):
        self.target_actor += self.tau * (self.actor - self.target_actor)
        self.target_critic += self.tau * (self.critic - self.target_critic)

print("DDPG: 确定性策略+行动者-评论家框架")

---
## 2. 目标网络

In [ ]:
class ReplayBuffer:
    """经验回放"""
    def __init__(self, capacity=10000):
        self.buffer = []
        self.capacity = capacity
    def push(self, transition):
        if len(self.buffer) >= self.capacity: self.buffer.pop(0)
        self.buffer.append(transition)
    def sample(self, batch_size):
        return random.sample(self.buffer, batch_size)

print("目标网络: 软更新确保训练稳定性")

---
## 3. 编程题

### 编程题1：实现DDPG算法在连续控制环境中的训练

In [ ]:
class DDPGAgent:
    """DDPG智能体"""
    def __init__(self, n_states, n_actions, tau=0.005):
        self.actor = np.random.randn(n_states) * 0.01
        self.critic = np.random.randn(n_states, n_actions) * 0.01
        self.target_actor = self.actor.copy()
        self.target_critic = self.critic.copy()
        self.tau = tau
        self.gamma = 0.99
    
    def get_action(self, state, noise=0.1):
        action = np.tanh(np.dot(state, self.actor))
        return np.clip(action + np.random.randn() * noise, -1, 1)
    
    def get_q(self, state, action):
        return np.dot(state, self.critic) + action
    
    def update(self, s, a, r, ns, done, alpha_a=0.001, alpha_c=0.01):
        target = r + self.gamma * (0 if done else self.get_q(ns, np.tanh(np.dot(ns, self.target_actor))))
        q_loss = (target - self.get_q(s, a)) ** 2
        self.critic -= alpha_c * q_loss * (s[:, None] if len(s.shape) > 1 else s)
        
        actor_loss = -self.get_q(s, np.tanh(np.dot(s, self.actor)))
        self.actor -= alpha_a * actor_loss * s
        
        self.target_actor += self.tau * (self.actor - self.target_actor)
        self.target_critic += self.tau * (self.critic - self.target_critic)

class ContinuousEnv:
    """连续动作Pendulum环境"""
    def __init__(self):
        self.state = np.random.randn(2) * 0.1
        self.max_torque = 2.0
    def reset(self): self.state = np.random.randn(2) * 0.1; return self.state.copy()
    def step(self, action):
        action = np.clip(action[0], -1, 1) * self.max_torque
        self.state += np.array([self.state[1], -self.state[0] + action]) * 0.05
        cost = self.state[0] ** 2 + 0.1 * self.state[1] ** 2 + 0.01 * action ** 2
        done = abs(self.state[0]) > 10
        return self.state.copy(), -cost, done

def train_ddpg():
    """训练DDPG"""
    agent = DDPGAgent(n_states=2, n_actions=1)
    buffer = ReplayBuffer(capacity=5000)
    env = ContinuousEnv()
    costs_history = []
    
    for ep in range(100):
        s = env.reset()
        total_cost = 0
        
        for _ in range(100):
            a = agent.get_action(s)
            ns, r, done = env.step(a)
            buffer.push((s, a, r, ns, done))
            s = ns
            total_cost += r
            
            if len(buffer) >= 32:
                for _ in range(10):
                    batch = buffer.sample(32)
                    for s_, a_, r_, ns_, done_ in batch:
                        agent.update(s_, a_, r_, ns_, done_)
        
        costs_history.append(total_cost)
    
    print("DDPG训练结果:")
    print(f"  最终回报: {costs_history[-1]:.2f}")
    print(f"  平均回报(最后10局): {np.mean(costs_history[-10:]):.2f}")

train_ddpg()

---

### 编程题2：实现DDPG与随机策略的对比实验

In [ ]:
class RandomAgent:
    """随机策略智能体"""
    def get_action(self, state, noise=0.1):
        return np.random.randn()
    def update(self, *args): pass

def compare_ddpg_random():
    """对比DDPG与随机"""
    ddpg = DDPGAgent(n_states=2, n_actions=1)
    random_agent = RandomAgent()
    env = ContinuousEnv()
    
    ddpg_rewards, random_rewards = [], []
    
    for agent in [ddpg, random_agent]:
        buffer = ReplayBuffer(capacity=5000) if hasattr(agent, 'update') else None
        
        for ep in range(50):
            s = env.reset()
            total = 0
            for _ in range(100):
                a = agent.get_action(s)
                ns, r, done = env.step(a)
                if buffer: buffer.push((s, a, r, ns, done))
                s = ns
                total += r
                if done: break
            
            if buffer and len(buffer) >= 32:
                for _ in range(10):
                    batch = buffer.sample(32)
                    for s_, a_, r_, ns_, done_ in batch:
                        agent.update(s_, a_, r_, ns_, done_)
        
        if agent == ddpg: ddpg_rewards.append(total)
        else: random_rewards.append(total)
    
    print("DDPG vs 随机对比:")
    print(f"  DDPG回报: {np.mean(ddpg_rewards):.2f}")
    print(f"  随机回报: {np.mean(random_rewards):.2f}")

compare_ddpg_random()

---

### 编程题3：实现带噪声探索的DDPG

In [ ]:
class OrnsteinUhlenbeckNoise:
    """Ornstein-Uhlenbeck探索噪声"""
    def __init__(self, size, mu=0, theta=0.15, sigma=0.2):
        self.size = size
        self.mu = mu
        self.theta = theta
        self.sigma = sigma
        self.x = np.zeros(size)
    def reset(self): self.x = np.zeros(self.size)
    def sample(self):
        self.x += self.theta * (self.mu - self.x) + self.sigma * np.random.randn(self.size)
        return self.x

class NoisyDDPG:
    """带OU噪声的DDPG"""
    def __init__(self, n_states):
        self.actor = np.random.randn(n_states) * 0.1
        self.critic = np.random.randn(n_states) * 0.1
        self.noise = OrnsteinUhlenbeckNoise(n_states)
    
    def get_action(self, state):
        action = np.tanh(np.dot(state, self.actor))
        return np.clip(action + self.noise.sample(), -1, 1)
    
    def update(self, s, a, r, ns, done):
        alpha = 0.001
        target = r + 0.99 * (0 if done else np.dot(ns, self.target_critic))
        q_loss = (target - np.dot(s, self.critic)) ** 2
        self.critic -= alpha * q_loss * s
        self.actor -= alpha * -np.dot(s, self.actor) * s

def test_noisy_ddpg():
    """测试带噪声DDPG"""
    agent = NoisyDDPG(n_states=2)
    env = ContinuousEnv()
    rewards_history = []
    
    for ep in range(50):
        agent.noise.reset()
        s = env.reset()
        total = 0
        
        for _ in range(100):
            a = agent.get_action(s)
            ns, r, done = env.step(a)
            agent.update(s, a, r, ns, done)
            s = ns
            total += r
            if done: break
        rewards_history.append(total)
    
    print("OU噪声DDPG:")
    print(f"  最终回报: {rewards_history[-1]:.2f}")

test_noisy_ddpg()

print("\n第16章 DDPG算法 - 编程题完成!")